# 1 – Indexing: PDF-Dokumente in eine Vektordatenbank überführen

Dieses Notebook führt die **Indexing-Pipeline** durch:

1. **Laden** – PDFs und Word-Dokumente werden eingelesen
2. **Chunking** – Der Text wird in überlappende Abschnitte zerlegt
3. **Embedding** – Jeder Chunk wird in einen Vektor umgewandelt
4. **Speichern** – Die Vektoren landen in einer lokalen ChromaDB

> **Hinweis:** Dieses Notebook muss nur einmal (oder bei neuen Dokumenten) ausgeführt werden.  
> Das RAG-Notebook (Notebook 2) liest dann die fertige Datenbank.

## Imports & Konfiguration

In [ ]:
import os
import glob
import shutil

from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
# --- KONFIGURATION ---

# Pfade
PDF_SOURCE_DIR = "./documents"      # Ordner mit den Quell-Dokumenten (PDF + DOCX)
DB_DIR         = "./chroma_db"       # Speicherort der Vektordatenbank
MODEL_PATH     = "./models"          # Lokaler Cache für das Embedding-Modell

# Embedding-Modell
EMBEDDING_MODEL = "intfloat/multilingual-e5-large-instruct"

# Chunking-Parameter
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128   # ~25 % Überlappung – wichtig bei langen deutschen Sätzen

## Embedding-Modell vorbereiten

Das Modell `multilingual-e5-large-instruct` erwartet **spezifische Präfixe**:  
- Dokumente (beim Indexieren): `"passage: ..."`  
- Suchanfragen (beim Retrieval): `"query: ..."`  

Ohne diese Präfixe arbeitet das Modell deutlich unter seinem Niveau.  
Da LangChain diese Präfixe nicht automatisch setzt, bauen wir einen dünnen Wrapper:

In [ ]:
class E5Embeddings(HuggingFaceEmbeddings):
    """Wrapper, der die vom E5-Modell erwarteten Präfixe automatisch setzt."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


embeddings = E5Embeddings(
    model_name=EMBEDDING_MODEL,
    cache_folder=MODEL_PATH,
)

## Schritt 1 – Dokumente laden (PDF & DOCX)

In [ ]:
# Mapping: Dateiendung → passender LangChain-Loader
LOADERS = {
    ".pdf":  PyMuPDFLoader,
    ".docx": Docx2txtLoader,
}


def load_documents(source_dir: str):
    """Lädt alle unterstützten Dokumente (PDF, DOCX) aus einem Verzeichnis."""
    all_docs = []
    file_count = 0

    for ext, loader_cls in LOADERS.items():
        for filepath in glob.glob(os.path.join(source_dir, f"*{ext}")):
            loader = loader_cls(filepath)
            all_docs.extend(loader.load())
            print(f"  📄 {os.path.basename(filepath)}")
            file_count += 1

    print(f"\n✅ {len(all_docs)} Seiten/Abschnitte aus {file_count} Dokumenten geladen.")
    return all_docs

## Schritt 2 – Text in Chunks aufteilen

In [ ]:
def split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Zerlegt Dokumente in überlappende Text-Chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    print(f"✅ {len(chunks)} Chunks erzeugt (Größe: {chunk_size}, Overlap: {chunk_overlap})")
    return chunks

## Schritt 3 – Embeddings erstellen & in ChromaDB speichern

In [ ]:
def create_vectorstore(chunks, embedding_fn, db_dir=DB_DIR):
    """Erzeugt Embeddings und speichert sie in einer lokalen ChromaDB."""
    # Alten Index löschen, damit keine veralteten Vektoren drin bleiben
    if os.path.exists(db_dir):
        shutil.rmtree(db_dir)
        print("🗑️  Alter Index gelöscht.")

    print("🧠 Erstelle Embeddings und speichere in ChromaDB...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_fn,
        persist_directory=db_dir,
    )
    print(f"✨ Fertig! Datenbank gespeichert in '{db_dir}'")
    return vectorstore

## Pipeline ausführen

In [ ]:
# --- PIPELINE ---
print("🚀 Starte Indexing-Pipeline...\n")

docs   = load_documents(PDF_SOURCE_DIR)
chunks = split_documents(docs)
db     = create_vectorstore(chunks, embeddings)

print(f"\n🎯 Nächster Schritt: Notebook 2 (RAG) öffnen und Fragen stellen.")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"MKL verfügbar: {torch.backends.mkl.is_available()}")
print(f"Threads: {torch.get_num_threads()}")